# Lesson 1 — The Acme Retail Incident

> **Case study.** On a Friday afternoon, *Acme Retail* deployed a new 
> AI "refund-bot" to handle customer service tickets. The bot was given 
> direct access to the refund API. Within **47 minutes**, an automated 
> abuse script issued **312 refund requests** averaging $155 each.
> 
> Total loss before the bot was disabled: **$48,200**.

**What went wrong?** The refund tool had no guardrails. The LLM cheerfully 
called `process_refund(order_id, amount)` whenever the user asked.

In this workshop you will build the missing safety layer using **Open Policy 
Agent (OPA)** wired into **Microsoft Agent Framework** tool calls.

## Learning objectives
By the end of this notebook you will be able to:
1. Reproduce the unguarded refund-bot incident.
2. Run the same scenario through a policy-enforced agent and see OPA stop it.
3. Read the audit log entry that documents the denial.


## 0. Prerequisites — make sure OPA is running

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
os.chdir(ROOT)               # so ./logs/policy-audit.jsonl lands in the standard location
sys.path.insert(0, str(ROOT))
from src.opa import OPAClient
opa = OPAClient()
assert opa.health_check(), 'OPA server unreachable on http://localhost:8181. Run: docker-compose up -d opa'
print('OPA reachable ✓   cwd =', pathlib.Path.cwd())

OPA reachable ✓   cwd = /home/anirudh/Projects/Rogers/agent-policy-opa


## 1. The unguarded refund function (what Acme deployed)

No checks. No limits. The LLM said *refund $349.98*, so $349.98 went out.


In [2]:
from src import mock_data

def unguarded_refund(order_id: str, amount: float) -> dict:
    order = mock_data.get_order(order_id)
    return {'order_id': order_id, 'refund_amount': amount,
            'status': 'refunded', 'original_amount': order.amount}

# Scenario A — cashier-level bot issuing a $349.98 refund
print(unguarded_refund('ORD-001', 349.98))
# Scenario B — refund without a receipt, 45 days late
print(unguarded_refund('ORD-002', 89.99))

{'order_id': 'ORD-001', 'refund_amount': 349.98, 'status': 'refunded', 'original_amount': 349.98}
{'order_id': 'ORD-002', 'refund_amount': 89.99, 'status': 'refunded', 'original_amount': 89.99}


👆 Two refunds processed in microseconds, no questions asked. This is exactly 
how Acme lost $48K.


## 2. The same scenarios through a *policy-enforced* agent

We swap `unguarded_refund` for `ReturnsAgent.invoke_tool('process_refund', ...)`. 
Under the hood, every call now passes through OPA-A (authorization) and 
OPA-B (behavior) before the refund logic runs.


In [3]:
from src.agents import ReturnsAgent
from src.models import AgentRole

bot = ReturnsAgent(role=AgentRole.CASHIER)

print('Scenario A — $349.98 refund as cashier:')
print(bot.invoke_tool('process_refund', {'order_id': 'ORD-001', 'amount': 349.98}))
print()
print('Scenario B — $89.99 no-receipt 45-day-old refund:')
print(bot.invoke_tool('process_refund', {'order_id': 'ORD-002', 'amount': 89.99}))

Scenario A — $349.98 refund as cashier:
{'ok': False, 'denied_by': 'OPA-A', 'reason': 'authorization denied'}

Scenario B — $89.99 no-receipt 45-day-old refund:
{'ok': False, 'denied_by': 'OPA-A', 'reason': 'authorization denied'}


Both calls return `ok: False`. Notice:
* Scenario A is **denied by OPA-A** — the cashier role only allows refunds ≤ $100.
* Scenario B is denied by **OPA-A first** (45 days > 30-day window), but if a 
  more privileged role tried it, **OPA-B** would still catch the no-receipt rule.


## 3. What just happened? The runtime architecture

```
  user message
       │
       ▼
  ┌──────────────┐  picks tool   ┌─────────────────────────┐
  │  LLM (GPT-4o) │ ────────────▶ │  @tool process_refund   │
  └──────────────┘                └────────────┬────────────┘
                                               │ self.enforce(...)
                                               ▼
                                  ┌──────────────────────────┐
                                  │ OPA-A  authorization.rego │
                                  │ OPA-B  behavior.rego      │
                                  └────────────┬─────────────┘
                                               │
                          allow ──────────────┼────────────── deny
                            │                                    │
                            ▼                                    ▼
                      run business               return {ok: False,
                       logic                     denied_by: 'OPA-A',
                                                   reason: '...'}
```


## 4. Every denial is audited

In [4]:
import json
audit_path = ROOT / 'logs' / 'policy-audit.jsonl'
lines = audit_path.read_text().strip().splitlines()[-4:]
for line in lines:
    entry = json.loads(line)
    print(f"{entry['timestamp'][:19]}  {entry['agent_id']:18}  "
          f"{entry['policy_path']:22}  allow={entry['allow']}  {entry['reason'] or ''}")

2026-05-28T21:37:48  returns-001         retail/authorization    allow=False  
2026-05-28T21:37:48  returns-001         retail/authorization    allow=False  
2026-05-28T21:49:11  returns-001         retail/authorization    allow=False  
2026-05-28T21:49:11  returns-001         retail/authorization    allow=False  


## Recap
* Without policy, the LLM can call any tool with any argument.
* Wrapping tools with OPA gives **role-based** and **behaviour-based** guardrails 
  that the LLM cannot bypass.
* Every decision (allow or deny) is written to `logs/policy-audit.jsonl` for 
  forensics.

**Next:** [02 — The Customer Data Leak](./02-opa-authorization.ipynb) — 
deep dive into OPA-A authorization.
